In [2]:
import warnings
from numba.core.errors import NumbaWarning

warnings.simplefilter('ignore', category=NumbaWarning)

In [3]:
import IPython.display as ipd
import torch
from torch.utils.data import DataLoader

import commons
import utils
from data_utils import TextAudioSpeakerLoader, TextAudioSpeakerCollate
from models import SynthesizerTrn
from text.symbols import symbols
from text import text_to_sequence


def get_text(text, hps):
    text_norm = text_to_sequence(text, hps.data.text_cleaners)
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    text_norm = torch.LongTensor(text_norm)
    return text_norm

DEBUG:matplotlib:matplotlib data path: /home/yash7/.local/lib/python3.12/site-packages/matplotlib/mpl-data
DEBUG:matplotlib:CONFIGDIR=/home/yash7/.config/matplotlib
DEBUG:matplotlib:interactive is False
DEBUG:matplotlib:platform is linux
DEBUG:matplotlib:CACHEDIR=/home/yash7/.cache/matplotlib
DEBUG:matplotlib.font_manager:Using fontManager instance from /home/yash7/.cache/matplotlib/fontlist-v390.json


/home/yash7/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO:numexpr.utils:NumExpr defaulting to 16 threads.
DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.piping.pipe(['renderer', 'formatter', 'neato_no_op', 'quiet'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.rendering.render(['renderer', 'formatter', 'neato_no_op', 'quiet'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.unflattening.unflatten(['stagger', 'fanout', 'chain', 'encoding'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.backend.viewing.view(['quiet'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.quoting.quote(['is_html_string', 'is_valid_id', 'dot_keywords', 'endswith_odd_number_of_backslashes', 'escape_unescaped_quotes'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.quoting.a_list(['kwargs', 'attributes'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.quoting.attr_list(['kwargs', 'attributes'])
DEBUG:graphviz._tools:deprecate positional args: graphviz.dot.Dot.clear(

[NeMo W 2025-04-25 05:32:08 nemo_logging:405] /home/yash7/.local/lib/python3.12/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
      warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
    


DEBUG:torio._extension.utils:Loading FFmpeg6
DEBUG:torio._extension.utils:Failed to load FFmpeg6 extension.
Traceback (most recent call last):
  File "/home/yash7/.local/lib/python3.12/site-packages/torio/_extension/utils.py", line 116, in _find_ffmpeg_extension
    ext = _find_versionsed_ffmpeg_extension(ffmpeg_ver)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yash7/.local/lib/python3.12/site-packages/torio/_extension/utils.py", line 108, in _find_versionsed_ffmpeg_extension
    _load_lib(lib)
  File "/home/yash7/.local/lib/python3.12/site-packages/torio/_extension/utils.py", line 94, in _load_lib
    torch.ops.load_library(path)
  File "/home/yash7/.local/lib/python3.12/site-packages/torch/_ops.py", line 1357, in load_library
    ctypes.CDLL(path)
  File "/home/yash7/miniconda3/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libavutil.so.58: cannot ope

## LJ Speech

In [ ]:
hps = utils.get_hparams_from_file("./configs/ljs_base.json")

In [ ]:
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model).cuda()
_ = net_g.eval()

_ = utils.load_checkpoint("./G_141000.pth", net_g, None)

/home/yash7/.local/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


INFO:root:Loaded checkpoint './G_141000.pth' (iteration 180)


In [ ]:
texts = ["This is a generated sentence", "These were not present in the dataset", "Some words such as LJSpeech need pronunciations for individual letters"]

for text in texts:
    stn_tst = get_text(text, hps)
    with torch.no_grad():
        x_tst = stn_tst.cuda().unsqueeze(0)
        x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
        audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
    ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=False))

## wTIMIT

In [4]:
hps = utils.get_hparams_from_file("./configs/wtimit_192.json")

In [5]:
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    n_speakers=hps.data.n_speakers,
    **hps.model).cuda()
_ = net_g.eval()

_ = utils.load_checkpoint("vits_mod_decoder.pth", net_g, None)

INFO:root:Loaded checkpoint 'vits_mod_decoder.pth' (iteration 87)


In [8]:
stn_tst = get_text("VITS has a voice-conversion pipeline in the provided code. Maybe it can be used for directly converting from whisper to speech", hps)
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    print(x_tst.shape, x_tst_lengths)
    sid = torch.LongTensor([31]).cuda()
    audio = net_g.infer(x_tst, x_tst_lengths, sid=sid, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
print(audio.shape)
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=True))
# audio = net_g(x_tst, x_tst_lengths, sid=sid, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()

with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    print(x_tst.shape, x_tst_lengths)
    sid = torch.LongTensor([2]).cuda()
    audio = net_g.infer(x_tst, x_tst_lengths, sid=sid, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=True))

torch.Size([1, 271]) tensor([271], device='cuda:0')
torch.Size([1, 192, 613])
(156928,)


torch.Size([1, 271]) tensor([271], device='cuda:0')
torch.Size([1, 192, 611])


### Voice Conversion

In [6]:
lines = []
with open('./filelists/wtimit_val.txt.cleaned', 'r+') as readFile:
    lines = readFile.readlines()
lines = [l.replace('/ssd_scratch/cvit/yash/converted_wavs/normal/', '/home/yash7/vF/valFiles/') for l in lines]
lines = [l.replace('/wavs/', '/') for l in lines]
with open('./filelists/wtimit_val2.txt.cleaned', 'w') as readFile:
    readFile.writelines(''.join(lines))

In [12]:
dataset = TextAudioSpeakerLoader("./test.txt.cleaned", hps.data)
collate_fn = TextAudioSpeakerCollate()
loader = DataLoader(dataset, num_workers=8, shuffle=False,
    batch_size=1, pin_memory=True,
    drop_last=True, collate_fn=collate_fn)
data_list = list(loader)

In [13]:
with torch.no_grad():
    x, x_lengths, spec, spec_lengths, y, y_lengths, sid_src = [x.cuda() for x in data_list[0]]
    sid_tgt1 = torch.LongTensor([38]).cuda()
    sid_tgt2 = torch.LongTensor([40]).cuda()
    sid_tgt3 = torch.LongTensor([7]).cuda()
    audio1 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt1)[0][0,0].data.cpu().float().numpy()
    audio2 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt2)[0][0,0].data.cpu().float().numpy()
    audio3 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt3)[0][0,0].data.cpu().float().numpy()
    audio4 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_src)[0][0,0].data.cpu().float().numpy()   
# print(len(audio1), len(audio1[0]), len(audio1[0][0]))
print("Original SID: %d" % sid_src.item())
ipd.display(ipd.Audio(y[0].cpu().numpy(), rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt1.item())
ipd.display(ipd.Audio(audio1, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt2.item())
ipd.display(ipd.Audio(audio2, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt3.item())
ipd.display(ipd.Audio(audio3, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_src.item())
ipd.display(ipd.Audio(audio4, rate=hps.data.sampling_rate, normalize=True))

PostEnc Inp:  torch.Size([1, 513, 441]) tensor([441], device='cuda:0')
torch.Size([1, 192, 441]) torch.Size([1, 1, 112896])
PostEnc Inp:  torch.Size([1, 513, 441]) tensor([441], device='cuda:0')
torch.Size([1, 192, 441]) torch.Size([1, 1, 112896])
PostEnc Inp:  torch.Size([1, 513, 441]) tensor([441], device='cuda:0')
torch.Size([1, 192, 441]) torch.Size([1, 1, 112896])
PostEnc Inp:  torch.Size([1, 513, 441]) tensor([441], device='cuda:0')
torch.Size([1, 192, 441]) torch.Size([1, 1, 112896])
Original SID: 55


Converted SID: 38


Converted SID: 40


Converted SID: 7


Converted SID: 55


In [10]:
import librosa

from mel_processing import spectrogram_torch
audio, sr = librosa.load('./genWhispers/test1.wav', sr=24_000)
audio = torch.Tensor(audio).unsqueeze(0)
spec = spectrogram_torch(
    audio,
    n_fft = 1024,
    sampling_rate=24_000,
    hop_size=hps.data.hop_length,
    win_size=hps.data.win_length,
    center = False
)
specLengths = torch.LongTensor([spec.size(1)]).cuda()
spec = spec.cuda()
with torch.no_grad():
    sid_src = torch.LongTensor([0]).cuda()
    sid_tgt1 = torch.LongTensor([7]).cuda()
    sid_tgt2 = torch.LongTensor([53]).cuda()
    sid_tgt3 = torch.LongTensor([89]).cuda()
    audio1 = net_g.voice_conversion(spec, specLengths, sid_src=sid_src, sid_tgt=sid_tgt1)[0][0,0].data.cpu().float().numpy()
    audio2 = net_g.voice_conversion(spec, specLengths, sid_src=sid_src, sid_tgt=sid_tgt2)[0][0,0].data.cpu().float().numpy()
    audio3 = net_g.voice_conversion(spec, specLengths, sid_src=sid_src, sid_tgt=sid_tgt3)[0][0,0].data.cpu().float().numpy()
    audio4 = net_g.voice_conversion(spec, specLengths, sid_src=sid_src, sid_tgt=sid_src)[0][0,0].data.cpu().float().numpy()   
# print(len(audio1), len(audio1[0]), len(audio1[0][0]))
print("Original SID: %d" % sid_src.item())
ipd.display(ipd.Audio(audio[0].cpu().numpy(), rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt1.item())
ipd.display(ipd.Audio(audio1, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt2.item())
ipd.display(ipd.Audio(audio2, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_tgt3.item())
ipd.display(ipd.Audio(audio3, rate=hps.data.sampling_rate, normalize=True))
print("Converted SID: %d" % sid_src.item())
ipd.display(ipd.Audio(audio4, rate=hps.data.sampling_rate, normalize=True))

PostEnc Inp:  torch.Size([1, 513, 822]) tensor([513], device='cuda:0')
torch.Size([1, 192, 822]) torch.Size([1, 1, 210432])
PostEnc Inp:  torch.Size([1, 513, 822]) tensor([513], device='cuda:0')
torch.Size([1, 192, 822]) torch.Size([1, 1, 210432])
PostEnc Inp:  torch.Size([1, 513, 822]) tensor([513], device='cuda:0')
torch.Size([1, 192, 822]) torch.Size([1, 1, 210432])
PostEnc Inp:  torch.Size([1, 513, 822]) tensor([513], device='cuda:0')
torch.Size([1, 192, 822]) torch.Size([1, 1, 210432])
Original SID: 0


Converted SID: 7


Converted SID: 53


Converted SID: 89


Converted SID: 0
